<a href="https://colab.research.google.com/github/AmiraFaisal/ETEC2T/blob/main/ROUGE%2BBERTScore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install bert_score --quiet
!pip install rouge-score --quiet
import json
import os
import pandas as pd
from google.colab import drive
from bert_score import score
from rouge_score import rouge_scorer
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# mount Drive
drive.mount('/content/drive', force_remount=True)

# Verify paths exist
qwen_path = "/content/drive/MyDrive/EvaltheEvaluators/Qwen_Model/Qwen_Descriptions.json"
gemma_path = "/content/drive/MyDrive/EvaltheEvaluators/ChartGemma_Model/ChartGemma_Descriptions.json"
images_dir = "/content/drive/MyDrive/EvaltheEvaluators/images"
GT_dir = "/content/drive/MyDrive/EvaltheEvaluators/images/L2L3_captions.json"

# Check Qwen JSON
if os.path.exists(qwen_path):
    with open(qwen_path, 'r') as f:
        qwen = json.load(f)
    print(f"\nJSON loaded: {len(qwen)} samples")
    print(f"Sample: {qwen[0]}")
else:
    print(f"JSON not found: {qwen_path}")

# Check ChartGemma JSON
if os.path.exists(gemma_path):
    with open(gemma_path, 'r') as f:
        gemma = json.load(f)
    print(f"\nJSON loaded: {len(gemma)} samples")
    print(f"Sample: {gemma[0]}")
else:
    print(f"JSON not found: {gemma_path}")

# Check images
if os.path.exists(images_dir):
    all_files = os.listdir(images_dir)
    images = [f for f in all_files if f.lower().endswith(('.png'))]
    print(f"\nImages directory: {len(images)} files")
    print(f"Sample: {images}")
else:
    print(f"Images directory not found: {images_dir}")

# Check ground truths (L2L3_captions.json)
if os.path.exists(GT_dir):
    with open(GT_dir, 'r') as f:
        GT = json.load(f)
    print(f"\nground truths loaded: {len(GT)} samples")
    print(f"Sample: {GT[0]}")
else:
    print(f"JSON not found: {GT_dir}")

Mounted at /content/drive

JSON loaded: 21 samples
Sample: {'img_id': '3128', 'answer': "The graph depicts the infant mortality rate from 2009 to 2018 for Armenia (in deaths per 1,000 live births). The starting point is at approximately 16 deaths per 1,000 live births, which gradually decreases over time until it reaches around 7 by 2018. There's no significant increase or decrease observed during this period. Overall, there seems to have been a steady decline in the infant mortality rate within the given timeframe."}

JSON loaded: 21 samples
Sample: {'img_id': 3128, 'answer': 'The chart shows the infant mortality rate in Armenia from 2009 to 2019. The rate is measured in deaths per 1,000 live births. The chart shows that the rate has been steadily decreasing over the past decade. In 2009, the rate was around 15 deaths per 1,000 live births. By 2019, the rate had fallen to around 10 deaths per 1,000 live births. This represents a significant improvement in the health of infants in Arme

In [3]:
# prepare the already loaded data into maps
def createDict(data_list, id_key='img_id', text_key='answer'):
    """Create a dict that maps string-IDs to captions from a list of dicts."""
    return {str(item[id_key]): item[text_key] for item in data_list}

# GT has 'caption_L2L3' as the text key, qwen and gemma have 'answer' from their respective JSONs
human_map = createDict(GT, text_key="caption_L2L3")
qwen_map = createDict(qwen, text_key="answer")
gemma_map = createDict(gemma, text_key="answer")

# align the lists based on the *intersection* of image IDs present in all datasets
# since `caption_L2L3` from `GT` contains for all original charts including the selected 21
common_ids = sorted(list(set(human_map.keys()) & set(qwen_map.keys()) & set(gemma_map.keys())))
references = [human_map[i] for i in common_ids]
candidates_qwen = [qwen_map[i] for i in common_ids]
candidates_gemma = [gemma_map[i] for i in common_ids]

# ROUGE

In [4]:
# calculate ROUGE per question
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def get_individual_rouge_scores(candidates, refs):
    results = [scorer.score(r, c) for c, r in zip(candidates, refs)]
    # Multiply ROUGE scores by 10
    r1_scores = [res['rouge1'].fmeasure * 10 for res in results]
    r2_scores = [res['rouge2'].fmeasure * 10 for res in results]
    rL_scores = [res['rougeL'].fmeasure * 10 for res in results]
    return r1_scores, r2_scores, rL_scores

r1_q_scores, r2_q_scores, rL_q_scores = get_individual_rouge_scores(candidates_qwen, references)
r1_g_scores, r2_g_scores, rL_g_scores = get_individual_rouge_scores(candidates_gemma, references)

# BERTScore

In [5]:
# calculate BERTScore (Rescaled) per question
# which uses the RoBERTa-large model by default
_, _, F1_q_rescaled_tensor = score(candidates_qwen, references, lang="en", verbose=False, rescale_with_baseline=True)
_, _, F1_g_rescaled_tensor = score(candidates_gemma, references, lang="en", verbose=False, rescale_with_baseline=True)

# Multiply BERTScore F1 scores by 10
F1_q_rescaled_list = (F1_q_rescaled_tensor * 10).tolist()
F1_g_rescaled_list = (F1_g_rescaled_tensor * 10).tolist()

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

# save results

In [10]:
# create DataFrame with per-question results
per_question_results = {
    "img_id": common_ids,
    "Qwen_ROUGE-1": r1_q_scores,
    "Qwen_ROUGE-2": r2_q_scores,
    "Qwen_ROUGE-L": rL_q_scores,
    "Qwen_BERTScore-F1": F1_q_rescaled_list,
    "ChartGemma_ROUGE-1": r1_g_scores,
    "ChartGemma_ROUGE-2": r2_g_scores,
    "ChartGemma_ROUGE-L": rL_g_scores,
    "ChartGemma_BERTScore-F1": F1_g_rescaled_list
}

df_per_question = pd.DataFrame(per_question_results)

print("--- ROUGE and BERTScore Comparison (Per Question) ---")
print(df_per_question.to_string(index=False, float_format=lambda x: "{:.4f}".format(x)))

--- ROUGE and BERTScore Comparison (Per Question) ---
img_id  Qwen_ROUGE-1  Qwen_ROUGE-2  Qwen_ROUGE-L  Qwen_BERTScore-F1  ChartGemma_ROUGE-1  ChartGemma_ROUGE-2  ChartGemma_ROUGE-L  ChartGemma_BERTScore-F1
  1061        0.5714        0.0000        0.4286            -0.5760              0.2632              0.0000              0.2632                   0.2778
  1064        1.9048        0.1942        1.1429             0.1930              2.6829              0.2500              1.9512                   2.7166
  1253        2.1429        0.1818        1.2500             1.4985              1.4583              0.0000              0.8333                   0.7974
  1356        2.6168        0.1905        1.6822             2.7119              3.2911              0.7792              2.0253                   3.8843
  2114        1.0526        0.1183        0.8187             0.0612              2.1053              0.6452              1.6842                   1.3596
  2121        4.5161        

In [12]:
drive_base_path = '/content/drive/MyDrive/EvaltheEvaluators/evaluations/Factuality/'

# define metrics and their corresponding column names in df_per_question
metrics = {
    'ROUGE-1': ('Qwen_ROUGE-1', 'ChartGemma_ROUGE-1'),
    'ROUGE-2': ('Qwen_ROUGE-2', 'ChartGemma_ROUGE-2'),
    'ROUGE-L': ('Qwen_ROUGE-L', 'ChartGemma_ROUGE-L'),
    'BERTScore-F1': ('Qwen_BERTScore-F1', 'ChartGemma_BERTScore-F1')
}

# iterate through each metric to create and save individual csvs
for metric_name, (qwen_col, gemma_col) in metrics.items():
    # create new dataframe for the current metric
    df_metric = df_per_question[['img_id', qwen_col, gemma_col]].copy()

    # rename columns to 'qwen' and 'chartgemma'
    df_metric.rename(columns={qwen_col: 'Qwen', gemma_col: 'ChartGemma'}, inplace=True)

    file_name = f'{metric_name.replace(" ", "_")}_results.csv'
    full_file_path = os.path.join(drive_base_path, file_name)
    df_metric.to_csv(full_file_path, index=False)